# GPSseq · Install legacy Neurite and VoxelMorph

Run this notebook **after** `python -m pip install -r requirements.txt` in the GPSseq Python 3.9 environment.

GPSseq uses its own historical **Neurite** and **VoxelMorph** snapshots. The public PyPI packages labelled `0.2` are not interchangeable with these snapshots.

Steps: verify the environment and archive checksums → install **Neurite** → install **VoxelMorph** → verify `Constant` and `VxmAffineFeatureDetector` in a fresh Python process.

After successful verification, continue with `tutorials/02_prepare_allen_atlas.ipynb`.


## 1. Check Python environment and local archives

Use the `gpsseq` Jupyter kernel. The provided archives are checked against their recorded SHA256 digests before installation.


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as md
import subprocess
import sys

if sys.version_info[:2] != (3, 9):
    raise RuntimeError(
        f"GPSseq requires Python 3.9, but this kernel is {sys.version.split()[0]} "
        f"({sys.executable}). Select the gpsseq kernel."
    )


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if ((candidate / "requirements.txt").is_file()
            and (candidate / "brain_registration").is_dir()
            and (candidate / "third_party").is_dir()
            and (candidate / "scripts").is_dir()):
            return candidate
    raise FileNotFoundError(
        "GPSseq project root not found. Open this notebook from inside GPSseq."
    )


ROOT = find_repo_root()
NEURITE_ARCHIVE = ROOT / "third_party" / "neurite-gpsseq-legacy.zip"
VXM_ARCHIVE = ROOT / "third_party" / "voxelmorph-gpsseq-legacy.zip"
VERIFY_SCRIPT = ROOT / "scripts" / "verify_installation.py"

EXPECTED_SHA256 = {
    NEURITE_ARCHIVE: "4aada0bdcb05e92511d98cfa6b84a2ec018d893444686444debd95ea3788b44e",
    VXM_ARCHIVE: "06c4dff583805fbb2e74da6263470d4637ac97ee571376da5933f71ac0ec335c",
}

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("GPSseq project:", ROOT)
for name, expected in {"tensorflow": "2.16.1", "keras": "3.9.0"}.items():
    try:
        installed = md.version(name)
    except md.PackageNotFoundError:
        raise RuntimeError(f"{name} is not installed; first run pip install -r requirements.txt")
    print(f"{name}: {installed}")
    if installed != expected:
        raise RuntimeError(f"{name} must be {expected}, not {installed}")

for archive, expected_sha in EXPECTED_SHA256.items():
    if not archive.is_file():
        raise FileNotFoundError(f"Missing source archive: {archive}")
    actual_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    print(f"{archive.name} SHA256: {actual_sha}")
    if actual_sha != expected_sha:
        raise RuntimeError(f"Archive checksum mismatch: {archive.name}. Do not install.")

if not VERIFY_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing: {VERIFY_SCRIPT}")
print("All prerequisite checks passed.")


## 2. Install GPSseq-compatible Neurite

This source ZIP includes `neurite.tf.layers.Constant` and the Keras-compatible `MAE`/`MSE` aliases, so the former `patch_neurite.py` step is **not needed**.


In [ ]:
cmd = [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(NEURITE_ARCHIVE)]
print("Installing:", NEURITE_ARCHIVE.name)
subprocess.run(cmd, check=True)
print("Neurite installed.")


## 3. Install GPSseq-compatible VoxelMorph

Install the bundled historical TensorFlow VoxelMorph snapshot; **do not** replace it with the public PyPI `voxelmorph==0.2` distribution.


In [ ]:
cmd = [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(VXM_ARCHIVE)]
print("Installing:", VXM_ARCHIVE.name)
subprocess.run(cmd, check=True)
print("VoxelMorph installed.")


## 4. Verify both packages in a fresh process

The script checks package versions, `neurite.layers.Constant`, `VxmAffineFeatureDetector`, and construction of the affine model with input shape `(256, 256)`. It does **not** load pretrained weights.


In [ ]:
subprocess.run([sys.executable, str(VERIFY_SCRIPT)], check=True)


## 5. Continue to Allen Atlas preparation

If all checks pass, run `tutorials/02_prepare_allen_atlas.ipynb` next.

When replacing either archive with a newer revision, its checksum must be updated only after reviewing and testing the new source. If Neurite or VoxelMorph is reinstalled from PyPI, rerun this notebook to restore the GPSseq snapshots.
